# 02 — LLM-пайплайн на Qwen3-4B (Colab T4)

Реальные LLM-шаги: классификатор разделов, extractor, классификатор запроса, ответ. Проверка на dev-наборе и живой диалог. Нужен GPU (T4).

In [ ]:
import os, sys
if not os.path.exists("src/acctltm"):
    !git clone https://github.com/serge3emskov/accountant-llm-assistant.git
    os.chdir("accountant-llm-assistant")
!pip install -q pydantic rapidfuzz
sys.path.insert(0, "src")

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from acctltm import AccountantAssistant, MemoryStore, Normalizer
from acctltm.llm import QwenTransformersLLM
from acctltm.devcheck import load_jsonl, check_state
from acctltm.evaluation import eval_query_classifier

llm = QwenTransformersLLM()  # Qwen3-4B-Instruct-2507, 4-bit NF4
print(llm.name)

## Dev-набор с реальной моделью

Каждый диалог прогоняется через write-path с LLM; итоговая память сравнивается с эталоном.

In [ ]:
def run_llm_dialog(d):
    assistant = AccountantAssistant(llm, store=MemoryStore())
    assistant.write_path.process("u", d["turns"], d["id"])
    expect = {"facts": d["expected_facts"], "states": d["expected_states"],
              "data": d.get("expected_data", [])}
    return check_state(assistant.store, "u", expect)

dialogs = load_jsonl("data/dev_dialogs.jsonl")
errors = {d["id"]: run_llm_dialog(d) for d in dialogs}
passed = sum(1 for e in errors.values() if not e)
print("dev с LLM:", passed, "/", len(dialogs))
for did, e in errors.items():
    if e:
        print(did, e)

## Классификатор запроса: LLM против правил

In [ ]:
from acctltm.pipeline import QueryClassifier
queries = load_jsonl("data/dev_queries.jsonl")
clf = QueryClassifier(llm)
llm_res = eval_query_classifier(queries, classifier=lambda t: clf(t)[0])
rules_res = eval_query_classifier(queries)
print("LLM:", llm_res["accuracy"], "правила:", rules_res["accuracy"])

## Живой диалог

In [ ]:
assistant = AccountantAssistant(llm, store=MemoryStore())
session = assistant.session("company-1")
print(session.say("Мы на УСН доходы, работаем с Ромашкой как поставщик."))
session.close()
print(session.say("Можно оплатить Ромашке наличными 150 тысяч и принять расходы?"))

## Заметки

Метрики этого прогона зависят от модели и версии промптов. Для кэширования повторных прогонов используйте `CachedLLM`.